In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# نحمل البيانات اللي نظفناها في المرحلة الأولى
df = pd.read_csv('Ames_Housing_Cleaned.csv')

# 1. تحويل النصوص لأرقام (One-Hot Encoding)
# اخترنا عمودين "المنطقة" و "طريقة البيع" ونحولهم لأرقام عشان الكمبيوتر يفهمهم
df = pd.get_dummies(df, columns=['MS Zoning', 'Sale Condition'], drop_first=True) # [cite: 78]

# 2. ترتيب الجودة (Ordinal Encoding)
# هنا بنعطي أرقام لجودة البيت (مثلاً: مميز = 5، ضعيف = 1) بشكل منطقي ومرتب
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
df['Exter Qual Num'] = df['Exter Qual'].map(quality_map) # [cite: 79]

# 3. ابتكار ميزات جديدة (Domain Features)
# بنحسب "سعر القدم المربع" لأنه يعطينا فكرة أدق عن قيمة البيت من السعر الإجمالي
df['Price_per_SqFt'] = df['SalePrice'] / df['Gr Liv Area'] # [cite: 81, 82]

# 4. دمج ميزتين (Interaction Feature)
# بنضرب "الجودة" في "المساحة" لأن البيت الكبير وبجودة عالية أكيد قيمته مختلفة
df['Qual_x_Area'] = df['Overall Qual'] * df['Gr Liv Area'] # [cite: 83]

# 5. تحجيم البيانات (Scaling)
# بنخلي الأرقام الكبيرة (زي المساحة) والأرقام الصغيرة في نطاق واحد عشان ما تضيع الحسبة
scaler = StandardScaler()
df[['Gr Liv Area', 'Lot Area']] = scaler.fit_transform(df[['Gr Liv Area', 'Lot Area']]) # [cite: 80]

# نشوف شكل البيانات الجديد بعد "الهندسة" اللي سويناها
print("تمت إضافة الميزات بنجاح! عدد الأعمدة صار:", df.shape[1])
df.head()

تمت إضافة الميزات بنجاح! عدد الأعمدة صار: 90


,Order,PID,MS SubClass,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,Lot Config,...,MS Zoning_RL,MS Zoning_RM,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial,Exter Qual Num,Price_per_SqFt,Qual_x_Area
0,1,526301100,20,141.0,2.744381,Pave,IR1,Lvl,AllPub,Corner,...,True,False,False,False,False,True,False,3,129.830918,9936
1,2,526350040,20,80.0,0.187097,Pave,Reg,Lvl,AllPub,Inside,...,False,False,False,False,False,True,False,3,117.187500,4480
2,3,526351010,20,81.0,0.522814,Pave,IR1,Lvl,AllPub,Corner,...,True,False,False,False,False,True,False,3,129.420617,7974
3,4,526353030,20,93.0,0.128458,Pave,Reg,Lvl,AllPub,Corner,...,True,False,False,False,False,True,False,4,115.639810,14770
4,5,527105010,60,74.0,0.467348,Pave,IR1,Lvl,AllPub,Inside,...,True,False,False,False,False,True,False,3,116.574586,8145


In [4]:
# 6. تحويل الأعمدة الملتوية (Log Transform)
# بنعالج ميلان البيانات في عمود السعر عشان يكون توزيعه طبيعي أكثر
df['SalePrice_Log'] = np.log1p(df['SalePrice'])

# 7. تقسيم البيانات لمجموعات (Binning)
# بنقسم سنة البناء لثلاث فئات: قديم، متوسط، وجديد
df['House_Age_Category'] = pd.cut(df['Year Built'],
                                   bins=[0, 1950, 2000, 2010],
                                   labels=['Old', 'Recent', 'New'])

# 8. تنظيف الميزات الزايدة (Correlation)
# بنشوف لو فيه عمودين زي بعض بالضبط ونحذف واحد عشان ما نشوش النموذج
corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df.drop(columns=to_drop, inplace=True)


In [5]:
# حفظ ملف الميزات لاستخدامه في المرحلة الثالثة
df.to_csv('Ames_Housing_Features.csv', index=False)
print("الملف جاهز للمرحلة الثالثة!")

الملف جاهز للمرحلة الثالثة!
